**IN:** directory of pdfs we want to extract info from, chromadb vector databases for each of those pdfs

**OUT:** csv that follows the same format as the ERA training data

In [ ]:
# TODO expand the code to intake multiple pdfs from a folder instead of just one 

In [ ]:
# Import libraries
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# Setting the environment

load_dotenv() # get your OPENAI_API_KEY from the .env file

DATA_PATH = r"pdfs"
CHROMA_PATH = r"chromadb"
PAPER_ID = "BO1005"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=PAPER_ID)

pre_prompt = """ 
You are a helpful assistant. You answer questions about diet information in livestock management scientific literature. 
But you only answer based on knowledge I'm providing you. You don't use your internal knowledge and you don't make things up.
If you don't know the answer, just say: I don't know
"""

# TODO edit query to get desired response and so it works for all pdfs
user_query = """
Give a csv table for each diet in this paper. 
Each row will have an ingredient in the diet, the ingredient type, 
the amount of the ingredient, the units for that amount, 
whether the ingredient is dry, and whether it was fed ad libitum.
Ingredient types include Crop Byproduct, Crop Product, Forage Plants, Supplement, and Other Ingredients
"""
# user_query = input("What do you want to know about this paper?\n") # user query can also come from user input

In [ ]:
# get chunks from chromadb relevant to user_query
results = collection.query(
    query_texts=[user_query],
    n_results=4 # TODO change amount of results to minimize token usage while keeping good outputs
)
# print(results['documents']) 

# get OpenAI client
client = OpenAI()

# define system prompt with chromadb results 
system_prompt = f"""
{pre_prompt}
--------------------
The data:
{str(results['documents'])}
"""

# send query to llm
response = client.chat.completions.create(
    model="gpt-4o-mini", # TODO check which model is most cost effective
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_query} 
    ]
)

# print llm response
print("\n\n---------------------\n\n")
print(response.choices[0].message.content)

In [ ]:
# TODO output a csv following the ERA training data format that includes data for all input pdfs